## 1. Setup & Data Loading

In [2]:
import pandas as pd
import os
import tarfile

In [4]:
# Download and extract dataset
# !wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz -O imdb.tar.gz
with tarfile.open(r"C:\mindful-ai\sapient-ds\2025\reference\day-04\classification-exercise\imdb.tar..gz", "r:gz") as tar:
    tar.extractall()

C:\Users\mindf\AppData\Local\Temp\ipykernel_7276\3548069731.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


In [5]:
# Load into a pandas DataFrame
def load_imdb(path, label):
    files = [(os.path.join(path, f), label) for f in os.listdir(path)]
    data = []
    for filepath, lbl in files:
        with open(filepath, encoding='utf-8') as f:
            data.append({'review': f.read(), 'sentiment': lbl})
    return pd.DataFrame(data)

In [6]:
train_pos = load_imdb('aclImdb/train/pos', 1)
train_neg = load_imdb('aclImdb/train/neg', 0)
test_pos  = load_imdb('aclImdb/test/pos', 1)
test_neg  = load_imdb('aclImdb/test/neg', 0)

df_train = pd.concat([train_pos, train_neg]).reset_index(drop=True)
df_test  = pd.concat([test_pos, test_neg]).reset_index(drop=True)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (25000, 2)
Test shape: (25000, 2)


## 2. Text cleaning and feature engineering

In [ ]:
import re
from textblob import TextBlob

def clean_text(text):
    text = text.lower()
    text = re.sub('<.*?>', '', text)
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_train['clean_review'] = df_train['review'].apply(clean_text)
df_test['clean_review']  = df_test['review'].apply(clean_text)

# Engineering features
for df in [df_train, df_test]:
    df['rev_len']   = df['clean_review'].apply(len)
    df['word_count']= df['clean_review'].apply(lambda s: len(s.split()))
    df['excl_cnt']  = df['review'].apply(lambda s: s.count('!'))
    df['sentiment_score'] = df['clean_review'].apply(lambda s: TextBlob(s).sentiment.polarity if s else 0)

df_train[['clean_review','rev_len','word_count','excl_cnt','sentiment_score']].head()


## 3. Prepare features and labels

In [ ]:
feature_cols = ['rev_len','word_count','excl_cnt','sentiment_score','clean_review']
X_train = df_train[feature_cols]
X_test  = df_test[feature_cols]
y_train = df_train['sentiment']
y_test  = df_test['sentiment']


## 4. Pipelines: Numeric and text

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

numeric_feats = ['rev_len','word_count','excl_cnt','sentiment_score']
text_feat = 'clean_review'

num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('svd', TruncatedSVD(n_components=100, random_state=42))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_feats),
    ('text', text_pipeline, text_feat)
])


## 5. Model Training 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Logistic Regression
pipe_lr = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


In [ ]:
# Random Forest
pipe_rf = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


## 7. Hyperparameter Tuning (GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_lr = {'clf__C': [0.1,1,10]}
gs_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=3, scoring='accuracy', n_jobs=-1)
gs_lr.fit(X_train, y_train)
print("Best LR params:", gs_lr.best_params_, "Acc:", gs_lr.best_score_)

param_grid_rf = {'clf__n_estimators':[100,200], 'clf__max_depth':[None,10,20]}
gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=3, scoring='accuracy', n_jobs=-1)
gs_rf.fit(X_train, y_train)
print("Best RF params:", gs_rf.best_params_, "Acc:", gs_rf.best_score_)


## 8. Explainability: SHAP for Classification

In [ ]:
import shap
best = gs_rf.best_estimator_
explainer = shap.TreeExplainer(best['clf'])
X_test_trans = best['pre'].transform(X_test)
shap_values = explainer.shap_values(X_test_trans[:200])  # sample for speed
shap.summary_plot(shap_values, X_test_trans[:200], feature_names=numeric_feats + [f"svd_{i}" for i in range(100)])


## 9. Simple Prediction

In [ ]:
sample = pd.DataFrame([{
    'rev_len': 100, 'word_count': 20, 'excl_cnt': 1, 'sentiment_score': 0.5,
    'clean_review': "Amazing movie with stunning acting and compelling plot."
}])
pred = best.predict(sample)[0]
prob = best.predict_proba(sample)[0][pred]
print(f"Predicted sentiment: {'Positive' if pred==1 else 'Negative'} (prob={prob:.2f})")
